# 05 - Post-training: from "continues text" to "answers questions"  *(~4 minutes)*

> **Presenter script.** "Here's a thing that surprises people. A pre-trained
> model is not a chatbot and never was. It is an autocomplete engine. If you type
> a question, it will happily continue with *more questions*, because that's what
> documents full of questions look like. Turning autocomplete into an assistant
> is a separate training stage, and it's much cheaper than the first one."

> **post-training** - everything you do to a model *after* pre-training to make
> it behave the way you want. The first and most important step is **supervised
> fine-tuning (SFT)**: show it thousands of examples of "here is a request, here
> is a good response".

In [ ]:
# --- boilerplate: make `import minigpt` work no matter where Jupyter started ---
import pathlib
import sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "minigpt").is_dir())
sys.path.insert(0, str(ROOT))

import torch

torch.set_num_threads(4)  # plenty for a model this small; more threads is not faster
print("repo root:", ROOT)

In [ ]:
from minigpt import data
from minigpt import train as T
from minigpt.model import DEFAULT_BATCH_SIZE
from minigpt.plots import plot_curves, use_stream_style

import matplotlib.pyplot as plt

use_stream_style()

tokenizer = data.load_tokenizer()

In [ ]:
USE_PREBAKED = True    # False = fine-tune live (~40 seconds)
SFT_STEPS = 400

## 1. Ask the base model a question

Load the pre-trained model from notebook 02 and simply ask it something.

In [ ]:
base_model, _, _ = T.load_checkpoint("base")

questions = [
    "what goes in the pan first?",
    "how many does it serve?",
    "who followed mila home?",
]

for q in questions:
    raw = T.generate(base_model, tokenizer, data.PROMPT_TEMPLATE.format(question=q),
                     max_new_tokens=90, temperature=0.5, top_k=10, seed=3)
    print("-" * 76)
    print(repr(raw))
print("-" * 76)

It rambles. It does not stop. It may well start writing a new story. That is not
a fault - **nobody ever showed it what an answer looks like.** It is doing
exactly the job it was trained for: continuing text.

## 2. The format

Fine-tuning data is pairs. We glue each pair into one string using a fixed
template, so the model learns that `a: ` is its cue to speak:

```
q: when do i add the thyme?
a: add the thyme right at the end.
```

The trailing newline matters: it is the model's "I'm done" signal, which is what
lets us stop generation cleanly later.

In [ ]:
train_pairs = data.load_qa("sft")
val_pairs = data.load_qa("sft_val")

print(f"{len(train_pairs)} training pairs, {len(val_pairs)} held-out pairs\n")
for pair in train_pairs[:3]:
    prompt, answer = data.format_example(pair["question"], pair["answer"])
    print(repr(prompt + answer))

## 3. Loss masking - the one idea worth stopping on

The example above is a single string, so a naive setup would grade the model on
predicting **every** character of it - including the question.

We do not want that. We are not hiring the model to write questions; we are
hiring it to write answers. So we mark every prompt position as "do not grade"
(the value `-100`, which our loss function ignores).

> **loss masking** - hiding part of the sequence from the loss so the model is
> only scored on the part you care about.

`.` below = ignored, `^` = graded.

In [ ]:
T.preview_loss_mask(train_pairs[0], tokenizer, base_model.cfg.block_size)

Note that the padding at the end is masked too - otherwise the model would spend
its effort learning to predict blank space.

In [ ]:
x, y = T.build_sft_batches(train_pairs, tokenizer, base_model.cfg.block_size)
vx, vy = T.build_sft_batches(val_pairs, tokenizer, base_model.cfg.block_size)
print(f"inputs : {tuple(x.shape)}  (examples, tokens)")
print(f"targets: {tuple(y.shape)}  with {(y == -100).float().mean():.0%} of positions masked out")

## 4. Fine-tune

Note the learning rate: `0.001`, a third of what we used for pre-training. Fine-
tuning is a *nudge*, not a rebuild - and, as notebook 04 showed, big nudges
destroy what is already there.

This is fast because we only need a few hundred steps. In the real world,
pre-training costs millions of dollars and SFT costs hundreds.

In [ ]:
if USE_PREBAKED:
    sft_model, _, _ = T.load_checkpoint("sft")
    sft_history = T.load_history("sft")
    print("loaded the pre-baked fine-tuned model")
else:
    sft_model, _, _ = T.load_checkpoint("base")
    sft_history = T.train_sft(sft_model, x, y, val_x=vx, val_y=vy,
                              steps=SFT_STEPS, batch_size=DEFAULT_BATCH_SIZE,
                              learning_rate=1e-3, eval_every=50, seed=1337, name="sft")

In [ ]:
plot_curves(sft_history, title="Supervised fine-tuning loss (answer tokens only)")
plt.show()

## 5. Before and after

Same model weights, four hundred steps apart.

In [ ]:
T.compare_answers(base_model, sft_model, tokenizer, questions, seed=3)

In [ ]:
for q in ["what is by the long beach?", "how long does it keep?",
          "what can i cook with leek?", "how did toby feel?"]:
    print(f"Q: {q}")
    print(f"A: {T.ask(sft_model, tokenizer, q, seed=11)}")
    print()

Look at what changed and what did **not**.

**Changed:** the model now answers in one short sentence and *stops*. That is
the shape of a response, and it learned it in 400 steps.

**Did not change:** the model does not know any facts. Ask "who followed mila
home?" and it will confidently produce a grammatically perfect answer about a
completely different character. Fine-tuning taught it *how to answer*, not
*what is true*.

That distinction is the single most useful thing to understand about chat
models. The polish is a training stage. The knowledge came from pre-training -
and if it was not in the pre-training data, no amount of fine-tuning puts it
there.

## 6. The mindset shift: loss stops being the scoreboard

During pre-training, validation loss was a genuinely good proxy for "is this
model better?".

After fine-tuning, it is not. A model can have a lovely low loss and still be
useless, rude, or confidently wrong - because "the answer a human would prefer"
is not a thing cross-entropy can see.

So the evaluation changes shape:

| stage | how you judge it |
|---|---|
| pre-training | validation loss, and it is trustworthy |
| fine-tuning | loss tells you training is *working*; **you read the outputs** to know if it is *good* |
| preference tuning | humans (or a model trained on human judgements) compare two answers and pick one |

Practically: keep a fixed list of ~20 prompts, run them after every fine-tuning
run, and actually read the answers. It feels unscientific. It is the single most
useful evaluation most teams have.

In [ ]:
print(f"final SFT validation loss: {sft_history.val_loss[-1]:.3f}")
print()
print("...which tells you the training loop worked. It does not tell you the")
print("answers are good. For that, read the cell above. Out loud. On stream.")

## 7. What comes next: preference tuning (concept only)

Supervised fine-tuning teaches the model *a* good answer. It cannot teach it
which of two good answers is **better** - because it only ever sees one.

That is what **preference tuning** is for.

**How the data is collected.** Give the model a prompt, sample two answers, and
ask a human "which do you prefer?". You end up with triples:
`(prompt, chosen answer, rejected answer)`.

**RLHF** (Reinforcement Learning from Human Feedback) was the original recipe:
train a separate *reward model* to predict human preference, then use
reinforcement learning to push the language model towards high-reward answers.
It works, and it is fiddly - three models, an RL loop, lots of ways to go wrong.

**DPO** (Direct Preference Optimization) is the popular modern shortcut. It
skips the reward model and the RL entirely. The insight is that you can write
down a loss function directly on the triples that says:

> make the chosen answer more likely than the rejected one, **but** don't drift
> too far from the model you started with.

That second clause is the whole game. Without it, the model finds degenerate
answers that score well and read terribly. In DPO it appears as a term that
compares the model against a frozen copy of itself from before tuning - the same
"don't bulldoze what you already have" instinct we used as *replay* in notebook
04 and as a *lower learning rate* here.

**Why we're not implementing it.** DPO needs preference data, which needs humans
comparing answers, which is a whole afternoon and not a live demo. But you now
know exactly where it slots in:

```
pre-training  ->  continued pre-training  ->  supervised fine-tuning  ->  preference tuning
  (language)        (a specific domain)          (follow instructions)      (be preferred)
```

Every model you have ever chatted with went through some version of that
pipeline. You have now done the first three.

## Recap

* Pre-training gives you an autocomplete engine, not an assistant.
* SFT teaches the *shape* of a response: prompt in, answer out, then stop.
* **Loss masking** means grading the answer only, never the question.
* Use a **smaller learning rate** than pre-training - you are nudging, not
  rebuilding.
* After this stage, **read the outputs**. The loss is a progress bar, not a
  verdict.

**Next:** `06_wrap_up.ipynb` - the one-page reference to keep.